#### Remove unbalanced data from the datasets

In [17]:
import shutil
from collections import Counter
from pathlib import Path

In [18]:
project_root=Path("__file__").resolve().parents[1]
source_data=project_root/"data"
filtered_data=project_root/"data_7class"

In [19]:
# Reconfigure Yolo IDs
old_to_new={
    1:0, # Corrugation
    2:1, # Cracks
    3:2, # Flasking
    6:3, # Shelling
    7:4, # Spalling
    8:5, # Squts
    9:6, # Wheel-Burn
}
new_class_names={
    0: "Corrugation",
    1: "Cracks",
    2: "Flaking",
    3: "Shelling",
    4: "Spalling",
    5: "Squats",
    6: "Wheel-Burn"
}

image_extensions={
    ".jpg", ".jpeg", ".png",
    ".bmp", ".webp"
}

if filtered_data.exists():
    shutil.rmtree(filtered_data)

for split in ["train", "valid", "test"]:
    source_images=source_data/split/"images"
    source_labels=source_data/split/"labels"

    output_images=filtered_data/split/"images"
    output_labels=filtered_data/split/"labels"

    output_images.mkdir(parents=True, exist_ok=True)
    output_labels.mkdir(parents=True, exist_ok=True)

    kept_images=0
    dropped_images=0

    for image_path in source_images.iterdir():
        if image_path.suffix.lower() not in image_extensions:
            continue

        label_path=source_labels/f"{image_path.stem}.txt"
        if not label_path.exists():
            continue
        new_labels=[]
        with open(label_path, "r") as f:
            for line in f:
                if not line.strip():
                    continue
                parts=line.strip().split()
                old_class_id=int(parts[0])
                # Keep only 7 classes
                if old_class_id in old_to_new:
                    new_class_id=old_to_new[old_class_id]
                    # replace class ID
                    parts[0]=str(new_class_id)
                    new_labels.append(" ".join(parts))

        # keep images with at least one defect
        if new_labels:
            shutil.copy2(
                image_path, 
                output_images/image_path.name
            )   

            new_label_path=(
                output_labels/f"{image_path.stem}.txt"
            ) 

            new_label_path.write_text(
                "\n".join(new_labels)+"\n",
                encoding="utf-8"
            )

            kept_images+=1
        else:
            dropped_images+=1
    print(
        f"{split}: kept={kept_images}; dropped={dropped_images}"
    )






train: kept=6260; dropped=421
valid: kept=592; dropped=14
test: kept=291; dropped=9


#### Verifying the new data folder

In [20]:
for split in ["train","valid","test"]:
    label_dir=filtered_data/split/"labels"
    image_dir=filtered_data/split/"images"

    counts=Counter()

    for label_file in label_dir.glob("*.txt"):
        with open(label_file, "r") as f:
            for line in f:
                if line.strip():
                    class_id=int(line.split()[0])
                    counts[class_id]+=1
    print(f"\n{split.upper()}")
    print("-"*40)
    print("Images:", len(list(image_dir.iterdir())))

    for class_id, class_name in new_class_names.items():
        print(
            f"{class_id}: "
            f"{class_name:<15} "
            f"{counts[class_id]:>6}"
        )


TRAIN
----------------------------------------
Images: 6260
0: Corrugation       5241
1: Cracks            4199
2: Flaking           4190
3: Shelling          5501
4: Spalling          4336
5: Squats            3208
6: Wheel-Burn        3561

VALID
----------------------------------------
Images: 592
0: Corrugation        489
1: Cracks             422
2: Flaking            390
3: Shelling           476
4: Spalling           416
5: Squats             305
6: Wheel-Burn         296

TEST
----------------------------------------
Images: 291
0: Corrugation        298
1: Cracks             234
2: Flaking            157
3: Shelling           268
4: Spalling           193
5: Squats             141
6: Wheel-Burn         184
